# w9_pfc.ipynb — 分布式分片 softmax(Partial-FC,大规模数据解决方案)

User (2026-07-23):swin 证明了滑动窗口有效;现在把负样本场**按 worker 分片**
——K 个分片各持 1/K 的环,各自在自己的 1/K 里做 swin,每步把 K 个分片窗口
的**分区函数精确合并**成一个 CE。单进程里的 concat 与真·多节点每步 all_reduce
partial partition **逐位等价**,所以这一步先验证算法与数字:分片 softmax 能否
在**逐步 SGD 节奏**下恢复同步基线(.941/.691),从而填平异步 epoch-push 那
4% 缺口。每个分片的编码只碰自己的 1/K,是内存故事要的数据局部性。臂名
`swin168...icetf_g4096_pfc{K}`;数值确认后,多进程 torch.distributed 系统演示
是机械抬升。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"

# distributed sharded softmax over the validated swin arm.
ARM = "wcle_swin168step84loop2i2ce_icetf"
CAP, EPOCHS = 4096, 2000
PFC_SHARDS = 8          # K: each shard holds 1/K of the ring (~202 games),
#                         > SWIN_W=168, so the 168 window genuinely SLIDES
#                         inside each shard -- the faithful structure (SWIN_W
#                         in a bigger shard). K<=9 keeps shard > window at
#                         N=1613; K=10+ shrinks the shard below 168.
PFC_WINDOW = 0          # 0 = let SWIN_W (168) slide inside the shard (the
#                         natural design; no artificial cap). Set below the
#                         shard size only to force lower per-step coverage.
os.makedirs(OUT_DIR, exist_ok=True)
_w = PFC_WINDOW or 168
_shard = 1613 // PFC_SHARDS
print(f"PFC run: {ARM}@{CAP} {EPOCHS}ep, K={PFC_SHARDS} shards "
      f"(~{_shard} games each), window={min(_w, _shard)} slides inside "
      f"-> per-step union ~{PFC_SHARDS*min(_w, _shard)} negatives "
      f"(~{PFC_SHARDS*min(_w, _shard)/1613:.0%} coverage)")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy", "h5py"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy h5py
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Single-process K-shard run: dispatches to train_v4doc's PFC mode,
# then projection + zsbest fall out of the standard tail. The concat of
# K shard partitions == the multi-node per-step all_reduce (numerically
# identical); this validates the algorithm and the numbers first.
import os, subprocess, time
from pathlib import Path

logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
PSW = os.path.join(REPO, "Pod", "w9_ps_worker.py")
name = (f"w9_{ARM}_g{CAP}_pfc{PFC_SHARDS}"
        + (f"w{PFC_WINDOW}" if PFC_WINDOW else "") + "_fp")
done = Path(OUT_DIR) / f"tower_{name}_ep{EPOCHS}.npz"
if done.exists():
    print(f"[skip] {name} already at ep{EPOCHS}")
else:
    cmd = ["python", "-u", PSW, "--data-dir", DATA_DIR, "--out-dir", OUT_DIR,
           "--repo", REPO, "--arm", ARM, "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", "50",
           "--pfc-shards", str(PFC_SHARDS),
           "--pfc-window", str(PFC_WINDOW),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH]
    print(f"launch {name} on gpu {gpus[0]} ...", flush=True)
    t0 = time.time()
    with open(logd / f"{name}.log", "w") as fh:
        rc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0],
                                     PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")).returncode
    print(f"rc={rc} [{(time.time()-t0)/3600:.1f}h]", flush=True)


In [ ]:
# Readout: PFC (sharded softmax) vs sync swin scratch + async rotate arm.
import json
from pathlib import Path
for nm, lab in ((f"w9_{ARM}_g{CAP}_pfc{PFC_SHARDS}" + (f"w{PFC_WINDOW}" if PFC_WINDOW else ""),
                 f"PFC K={PFC_SHARDS} w={PFC_WINDOW or 'full'}"),
                (f"w9_{ARM}_g{CAP}", "sync swin scratch (ref)"),
                (f"w9_{ARM}_g{CAP}_psdc5shc30r50be", "async rotate no-far (ref)"),
                ("w9_wcle_i2ce_icetf_g4096", "i2ce full (ref)")):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    if p.exists():
        d = json.loads(p.read_text())
        print(f"{lab:26s} ep{d['best_ep']:>4} neu {d['nm_neutral']:.3f} "
              f"non {d['nm_noname']:.3f} tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    else:
        print(f"{lab:26s} (pending)")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
